<div dir="rtl">

# الدرس 19: بايز الساذج وتصنيف النصوص

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### من الصفر

</div>

In [ ]:
import math
from collections import Counter

train = [("free money", "spam"), ("free prize now", "spam"), ("money now", "spam"),
         ("lunch now", "ham"), ("meeting noon", "ham"), ("lunch noon", "ham")]

vocab = {w for text, _ in train for w in text.split()}
classes = ["spam", "ham"]
prior = {c: sum(1 for _, l in train if l == c) / len(train) for c in classes}
counts = {c: Counter(w for text, l in train if l == c for w in text.split()) for c in classes}
totals = {c: sum(counts[c].values()) for c in classes}

def log_score(text, c):
    s = math.log(prior[c])
    for w in text.split():
        s += math.log((counts[c][w] + 1) / (totals[c] + len(vocab)))   # تنعيم لابلاس
    return s

msg = "free now"
scores = {c: log_score(msg, c) for c in classes}
print("log scores:", {c: round(v, 3) for c, v in scores.items()})

# من اللوغاريتمات إلى احتمالات: نفس حيلة سوفتماكس المستقرة
m = max(scores.values())
exp = {c: math.exp(v - m) for c, v in scores.items()}
print("P(spam | msg) =", round(exp["spam"] / sum(exp.values()), 3))

<div dir="rtl">

### بمكتبة سايكت ليرن

</div>

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

texts = [t for t, _ in train]
labels = [1 if l == "spam" else 0 for _, l in train]

vec = CountVectorizer()
X = vec.fit_transform(texts)             # حقيبة الكلمات
print("vocabulary:", list(vec.get_feature_names_out()))
print(X.toarray())

nb = MultinomialNB(alpha=1.0).fit(X, labels)    # alpha = 1 يعني تنعيم لابلاس
print("P(ham), P(spam):", nb.predict_proba(vec.transform(["free now"])).round(3))

<div dir="rtl">

### على بيانات أكبر

</div>

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(3)
spam_words = ["free", "win", "prize", "money", "offer", "click", "now", "cash"]
ham_words = ["meeting", "lunch", "project", "report", "tomorrow", "team", "now", "call"]

def make_msg(words, other, k=6):
    picks = list(rng.choice(words, size=k)) + list(rng.choice(other, size=2))   # قليل من كلمات الفئة الأخرى
    return " ".join(picks)

texts = [make_msg(spam_words, ham_words) for _ in range(100)] + [make_msg(ham_words, spam_words) for _ in range(100)]
labels = [1] * 100 + [0] * 100
t_tr, t_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)

vec = CountVectorizer()
nb = MultinomialNB().fit(vec.fit_transform(t_tr), y_tr)
print("test accuracy:", nb.score(vec.transform(t_te), y_te))
print("example:", t_te[0], "->", nb.predict(vec.transform([t_te[0]]))[0])

<div dir="rtl">

## تمرين: رسالة جديدة

استخدم نفس الرسائل الست. صنّف الرسالة: lunch money.

1. احسب درجة كل فئة بيدك، مع التنعيم.
2. حوّلها إلى احتمال أن الرسالة مزعجة.
3. تحقق بالكود، بالدالة التي كتبناها من الصفر، وبالمكتبة.

</div>

In [ ]:
# اكتب حلّك هنا
